# Generate synthetic times

In [1]:
import numpy as np
import pandas as pd

In [2]:
CSV_PATH = "../data/times.csv"


np.random.seed(42)
n = 100_000

# Category and outcome labels to sample from
cats = ["Category 1", "Category 2", "Category 3", "Category 4"]
outcomes = ["See & Treat", "See & Convey ED", "See & Convey non ED"]

# Build the base dataframe
# np.random.choice with no p argument assigns equal probability to each option
df = pd.DataFrame({
    "ResponseCategoryGroupLevel2": np.random.choice(cats, n),
    "C0660_CallOutcomeDetail": np.random.choice(outcomes, n),
})

# Rough boundaries
durations = {
    "C0001_Mobilisation_Duration": [0, 100],
    "C0004_MobilisationToScene_Duration": [0, 1000],
    "C0007_OnScene_Duration": [0, 5000],
    "C0172_SceneToDestination_Duration": [0, 2000],
    "C0012_Handover_Duration": [0, 3000],
    "C0171_WrapUp_Duration": [0, 1000],
}

# Uniform sampling with varying average by response category
cat_shift = {
    "Category 1": 0.0,
    "Category 2": 0.15,
    "Category 3": 0.30,
    "Category 4": 0.45
}
for col, (lo, hi) in durations.items():
    # Uniform sampling within rough boundaries
    base = np.random.uniform(low=lo, high=hi, size=n)
    # Shift results by response category so each looks a little different
    shift = df["ResponseCategoryGroupLevel2"].map(cat_shift) * (hi - lo)
    df[col] = np.clip(base + shift, lo, hi).round(1)

In [3]:
df.head()

,ResponseCategoryGroupLevel2,C0660_CallOutcomeDetail,C0001_Mobilisation_Duration,C0004_MobilisationToScene_Duration,C0007_OnScene_Duration,C0172_SceneToDestination_Duration,C0012_Handover_Duration,C0171_WrapUp_Duration
0,Category 3,See & Treat,100.0,302.3,5000.0,1567.4,2391.2,446.9
1,Category 4,See & Convey ED,100.0,1000.0,2401.9,1722.4,3000.0,1000.0
2,Category 1,See & Convey ED,55.6,530.6,3486.0,508.1,1219.9,82.2
3,Category 3,See & Convey ED,100.0,1000.0,4134.6,1711.8,978.7,1000.0
4,Category 3,See & Treat,36.9,312.6,5000.0,1931.6,1069.5,744.1


In [4]:
df.to_csv(CSV_PATH, index=False)